# Model Training Framework 1 
#### Code for training three model architectures.

In [1]:
# Import necessary modules
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import os
from os import listdir
from os.path import isfile, join
from PIL import Image
import keras
from keras import layers
import csv
from tensorflow.keras.utils import to_categorical

2025-10-30 18:32:03.993888: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761849124.414945      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761849124.547382      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
# Read the data set 
eda = pd.read_csv('train_labels.csv')
eda

,id,label
0,f38a6374c348f90b587e046aac6079959adf3835,0
1,c18f2d887b7ae4f6742ee445113fa1aef383ed77,1
2,755db6279dae599ebb4d39a9123cce439965282d,0
3,bc3f0c64fb968ff4a8bd33af6971ecae77c75e08,0
4,068aba587a4950175d04c680d38943fd488d6a9d,0
...,...,...
220020,53e9aa9d46e720bf3c6a7528d1fca3ba6e2e49f6,0
220021,d4b854fe38b07fe2831ad73892b3cec877689576,1
220022,3d046cead1a2a5cbe00b2b4847cfb7ba7cf5fe75,0
220023,f129691c13433f66e1e0671ff1fe80944816f5a2,0


In [3]:
# Function to select a train set sample, because the 0/1 division is 60/40
# the sample is stratified this way
def sub_sample(frac_):
    zero_sample = int(len(eda)*frac_*0.6)
    eda_zero = eda[eda.label == 0]
    sample_eda_zero = eda_zero.sample(n = zero_sample, replace=False, random_state=52)

    # A sub sample of only one labels  
    one_sample = int(len(eda)*frac_*0.4)
    eda_one = eda[eda.label == 1]
    sample_eda_one = eda_one.sample(n = one_sample, replace=False, random_state=52)

    # Merge the sample data sets 
    frames = [sample_eda_zero, sample_eda_one]
    train_df = pd.concat(frames)

    train_df = train_df.sample(frac=1, random_state = 126)
    
    return train_df

In [4]:
# Apply sub_sample function
train_df = sub_sample(0.1)
train_df

,id,label
176288,1d122a55476973f3847a705c96817c1e1b5ede9d,0
155346,5d94c8b0f0645c894c0ffcb292bbceb70393ea20,0
48863,d505c8b4fc7d65596ff3a8e6197b3a2474b68371,0
175558,11f7a527b4cabfcc4d00b455c9ecb52fc0547feb,1
195137,36ddfecad2e2b330e7bc8ba8fde52ddd2300168e,0
...,...,...
177151,db01a27193d200bc6d626f22d40fa2a441ebe082,1
211211,b3ed317e4997ad9435d2a8f9aeddcdb16fff079f,0
28330,f2552d8e74f0c4ccead7af168b9c3d5e2ce94bce,0
5173,112e7f5aff9bb19cfe15e8646ba6a5bcf7dbe4fd,0


In [5]:
# Function to transform .tif file to an array and to one hot encode the label 
def to_array(train_df = train_df):

    # String of working directory
    train_dir = '/kaggle/input/histopathologic-cancer-detection/train'
    
    imarray_totaal = []
    # Loop to import training images n train_df (by file names) from working directory
    for j in train_df.id:
    
        im = Image.open(train_dir + '/' + j + '.tif')
        # Turn '.tif' file into array
        imarray = np.array(im)
        
        imarray_totaal.append(imarray)
    
    # Turn list into array
    imarray_totaal= np.array(imarray_totaal)
    
    # One hot encoding for the label: Turning one label (0/1) into two labels (Label_0 and Label_1) 
    num_classes = 2
    labels = to_categorical(train_df.label, num_classes=num_classes)
    
    # Giving imarray_totaal (x_train) and labels (y_train) familiar names 
    x_train = imarray_totaal
    y_train = labels

    return x_train, y_train

In [6]:
# Function to write model training results to an external location 
def write_away(name, model_results, path):

# Writing away the results 
    name = pd.DataFrame.from_dict(model_results)
    name.to_csv(path, index=False)
    return name

In [7]:
# Apply to_array
train_set = to_array()
x_train = train_set[0]
y_train = train_set[1]

### Model 1

In [8]:
num_classes = 2
input_shape = (96, 96, 3)

model = keras.Sequential(
    [keras.Input(shape = input_shape),
     layers.Conv2D(filters = 32, kernel_size=(3,3),activation = 'sigmoid'),
     layers.MaxPooling2D(pool_size=(2, 2)),
     layers.Flatten(),
     layers.Dense(num_classes, activation="softmax"),
    ]
    )
model.summary()

I0000 00:00:1761849509.231279      37 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1761849509.231985      37 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 94, 94, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 47, 47, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 70688)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │       141,378 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 142,274 (555.76 KB)

 Trainable params: 142,274 (555.76 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# training model 1
batch_size = 200
epochs = 20
model.compile(loss="categorical_crossentropy", optimizer="sgd", metrics=["accuracy", "auc"])

history = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

Epoch 1/20


I0000 00:00:1761849521.214682     101 service.cc:148] XLA service 0x7aa698007300 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1761849521.215851     101 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1761849521.215873     101 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1761849521.453288     101 cuda_dnn.cc:529] Loaded cuDNN version 90300


  7/100 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.4988 - auc: 0.5371 - loss: 19.6773

I0000 00:00:1761849524.771472     101 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 46ms/step - accuracy: 0.5158 - auc: 0.5267 - loss: 25.8441 - val_accuracy: 0.6025 - val_auc: 0.6025 - val_loss: 41.0164
Epoch 2/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.5297 - auc: 0.5282 - loss: 25.9764 - val_accuracy: 0.6025 - val_auc: 0.6025 - val_loss: 42.5910
Epoch 3/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.5301 - auc: 0.5301 - loss: 24.1625 - val_accuracy: 0.6025 - val_auc: 0.6025 - val_loss: 20.5835
Epoch 4/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.5220 - auc: 0.5231 - loss: 20.3982 - val_accuracy: 0.6025 - val_auc: 0.6025 - val_loss: 47.2044
Epoch 5/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.5278 - auc: 0.5270 - loss: 21.6627 - val_accuracy: 0.6025 - val_auc: 0.6025 - val_loss: 27.5774
Epoch 6/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.5291 - auc: 0.5354 - loss: 20.5417 - val_accuracy: 0.3975 - val_auc: 0.3975 - val_loss: 42.7931
Epoch 7/20
100/100 ━━━━━━━━━━━━━━

In [11]:
# Model 1: Writing away the results 
write_away('Simple_CNN', history.history, 'Simple_CNN.csv')

,accuracy,auc,loss,val_accuracy,val_auc,val_loss
0,0.515984,0.519213,26.532612,0.602453,0.602453,41.016365
1,0.522953,0.521891,25.784670,0.602453,0.602453,42.590958
2,0.523155,0.523769,22.092428,0.602453,0.602453,20.583513
3,0.525933,0.528947,20.064478,0.602453,0.602453,47.204380
4,0.524569,0.524511,20.616272,0.602453,0.602453,27.577425
5,0.525782,0.528600,20.394987,0.397547,0.397547,42.793072
6,0.530882,0.532719,19.369314,0.602453,0.602453,28.552139
7,0.538710,0.540832,18.400537,0.397547,0.397547,58.787315
8,0.523256,0.524442,30.518532,0.602453,0.602453,32.017765
9,0.536034,0.539475,23.344458,0.410722,0.412143,17.359217


### Model 2

In [12]:
num_classes = 2
input_shape = (96, 96, 3)

model1 = keras.Sequential(
    [keras.Input(shape = input_shape),
     layers.Conv2D(filters = 32, kernel_size=(3,3),activation = 'sigmoid'),
     layers.MaxPooling2D(pool_size=(2, 2)),
     layers.Conv2D(64, kernel_size=(3, 3), activation="sigmoid"),
     layers.MaxPooling2D(pool_size=(2, 2)),
     layers.Flatten(),
     layers.Dense(num_classes, activation="softmax"),
    ]
    )
model1.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 94, 94, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 47, 47, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 45, 45, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 30976)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │        61,954 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 81,346 (317.76 KB)

 Trainable params: 81,346 (317.76 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# training model 2
batch_size = 200
epochs = 20

model1.compile(loss="categorical_crossentropy", optimizer="sgd", metrics=["accuracy", "auc"])

history1 = model1.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

Epoch 1/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 10s 65ms/step - accuracy: 0.5314 - auc: 0.5396 - loss: 4.8377 - val_accuracy: 0.3975 - val_auc: 0.3974 - val_loss: 2.3823
Epoch 2/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.5401 - auc: 0.5620 - loss: 0.7848 - val_accuracy: 0.6025 - val_auc: 0.6039 - val_loss: 1.2250
Epoch 3/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - accuracy: 0.5964 - auc: 0.6033 - loss: 0.7077 - val_accuracy: 0.3975 - val_auc: 0.3881 - val_loss: 2.0897
Epoch 4/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - accuracy: 0.5850 - auc: 0.5801 - loss: 0.7508 - val_accuracy: 0.6025 - val_auc: 0.6582 - val_loss: 1.2201
Epoch 5/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.5956 - auc: 0.6022 - loss: 0.7128 - val_accuracy: 0.3975 - val_auc: 0.3566 - val_loss: 1.8771
Epoch 6/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.5803 - auc: 0.5875 - loss: 0.7279 - val_accuracy: 0.6025 - val_auc: 0.6624 - val_loss: 1.0971
Epoch 7/20
100/100 ━━━━━━━━━━━━━━

In [14]:
# Model 2: Writing away the results 
write_away('Basic_CNN', history1.history, 'Basic_CNN.csv')

,accuracy,auc,loss,val_accuracy,val_auc,val_loss
0,0.522903,0.537365,2.418547,0.397547,0.397366,2.382312
1,0.551285,0.575359,0.714407,0.602453,0.603898,1.224959
2,0.586890,0.598062,0.687219,0.397547,0.388072,2.089719
3,0.585778,0.591295,0.695997,0.602453,0.658151,1.220074
4,0.593152,0.602870,0.684799,0.397547,0.356619,1.877133
5,0.592344,0.598123,0.687036,0.602453,0.662376,1.097071
6,0.599616,0.601541,0.680755,0.602453,0.664198,1.096807
7,0.596889,0.600806,0.681332,0.602453,0.657897,1.065437
8,0.599465,0.605328,0.679700,0.602453,0.667640,1.050527
9,0.597697,0.617880,0.674912,0.602453,0.726846,1.129296


In [15]:
# Apply function sub_sample (20%) 
train_df = sub_sample(0.2)
train_df

,id,label
101866,8312c9bac5ab0dded0b79b0e8793ac4470727f40,0
104382,146db8073ee4fc6e64dac8cc8b835306ce4f00a5,1
123473,d6bfa926359cdffe8a770c4c6513322924825928,1
201216,9e2bb84236b7adcd4d245dd6ac9d573bea10204b,0
62800,1809061f44efe7f494c72da733ba50f6a5f054c9,0
...,...,...
110944,18b62ca10f10a13b9dcab6c377a69e3afbb4f716,0
118348,74e880c6deb43c4d0a31adba76becb1eebfaa813,1
28330,f2552d8e74f0c4ccead7af168b9c3d5e2ce94bce,0
5173,112e7f5aff9bb19cfe15e8646ba6a5bcf7dbe4fd,0


In [16]:
# Apply function to_array()
train_set = to_array()
x_train = train_set[0]
y_train = train_set[1]

### Model 3 

In [17]:
num_classes = 2
input_shape = (96, 96, 3)

model2 = keras.Sequential(
    [keras.Input(shape = input_shape),
     layers.Conv2D(filters = 32, kernel_size=(3,3),activation = 'sigmoid'),
     layers.MaxPooling2D(pool_size=(2, 2)),
     layers.Conv2D(64, kernel_size=(3, 3), activation="sigmoid"),
     layers.MaxPooling2D(pool_size=(2, 2)),
     layers.Flatten(),
     layers.Dense(num_classes, activation="softmax"),
    ]
    )
model2.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 94, 94, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 47, 47, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 45, 45, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 30976)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │        61,954 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 81,346 (317.76 KB)

 Trainable params: 81,346 (317.76 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
# training model 3
batch_size = 200
epochs = 20

model2.compile(loss="categorical_crossentropy", optimizer="sgd", metrics=["accuracy", "auc"])

history2 = model2.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

Epoch 1/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 8s 59ms/step - accuracy: 0.5298 - auc: 0.5405 - loss: 3.9475 - val_accuracy: 0.3975 - val_auc: 0.3390 - val_loss: 2.7819
Epoch 2/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.5693 - auc: 0.5758 - loss: 0.7947 - val_accuracy: 0.6025 - val_auc: 0.6069 - val_loss: 1.2657
Epoch 3/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.5944 - auc: 0.6009 - loss: 0.7093 - val_accuracy: 0.6025 - val_auc: 0.6302 - val_loss: 1.1668
Epoch 4/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.5999 - auc: 0.6030 - loss: 0.7023 - val_accuracy: 0.6025 - val_auc: 0.6707 - val_loss: 1.1526
Epoch 5/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.5926 - auc: 0.6006 - loss: 0.7032 - val_accuracy: 0.6025 - val_auc: 0.6558 - val_loss: 1.0803
Epoch 6/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.6041 - auc: 0.6163 - loss: 0.6909 - val_accuracy: 0.6025 - val_auc: 0.6464 - val_loss: 1.1681
Epoch 7/20
100/100 ━━━━━━━━━━━━━━━

In [19]:
# Model 3: Writing away the results 
write_away('Basic_CNN_20perc', history2.history, 'Basic_CNN_20perc.csv')

,accuracy,auc,loss,val_accuracy,val_auc,val_loss
0,0.538003,0.549295,1.862444,0.397547,0.338974,2.781864
1,0.580425,0.590821,0.706368,0.602453,0.606876,1.265699
2,0.590576,0.601318,0.684753,0.602453,0.630244,1.166794
3,0.599010,0.601818,0.682484,0.602453,0.670690,1.152603
4,0.593808,0.604799,0.680995,0.602453,0.655834,1.080333
5,0.599566,0.612951,0.676514,0.602453,0.646435,1.168056
6,0.598101,0.631015,0.673163,0.397547,0.372517,1.633631
7,0.596536,0.633414,0.672364,0.602453,0.621531,1.096624
8,0.592899,0.645974,0.661120,0.602453,0.631006,1.205126
9,0.593606,0.649369,0.655669,0.397547,0.482676,1.813294
